In [4]:
import requests
import logging
from logging.handlers import RotatingFileHandler
from datetime import datetime, time as dt_time 
import time as t
import talib
import pandas as pd
from talib import SMA, BBANDS, STOCH


In [5]:

# Configure logging with rotating file handler
log_file = "api/logs/optionChain.log"
handler = RotatingFileHandler(log_file, maxBytes=5000000, backupCount=5)  # 5 MB max per log file, keep 5 backups
formatter = logging.Formatter('%(asctime)s - %(levelname)s - %(message)s')
handler.setFormatter(formatter)

# Root logger
logger = logging.getLogger()
logger.setLevel(logging.DEBUG)
logger.addHandler(handler)

logger.info("Starting real-time data insertion script.")

def get_current_timestamp():
    now = datetime.now()
    return now.strftime("%Y-%m-%d %H:%M:%S") + f".{now.microsecond // 1000:03d}"



In [6]:
# Fetch and process data from API
class OptionChainFetcher:
    def fetch_data(self):
        # Read the access token from file
        try:
            with open('api/token/accessToken_OC.txt', 'r') as file:
                access_token = file.read().strip()
        except FileNotFoundError:
            logging.error("Access token file not found.")
            return {}

        url = 'https://api.upstox.com/v2/option/chain'
        params = {
            'instrument_key': 'BSE_INDEX|SENSEX',
            'expiry_date': '2024-09-13'
        }
        headers = {
            'Accept': 'application/json',
            'Authorization': f'Bearer {access_token}'
        }

        try:
            response = requests.get(url, params=params, headers=headers)
            response.raise_for_status()  # Raise an exception for HTTP errors
            logging.debug(f"Request URL: {response.url}")
            return response.json()
        except requests.RequestException as e:
            logging.error(f"Request failed: {e}")
            return {}
        
    def apply_indicators(self, df):
        # Ensure that the required columns are mapped correctly
        close_price = df['close_price']  # 'close' in talib maps to 'close_price'
        ltp = df['ltp']  # 'open' in talib maps to 'ltp'
        volume = df['volume']  # 'volume' in talib maps to 'volume'
        oi = df['oi']  # 'oi' should map directly to 'oi'

        # Apply the SMA indicator
        df['SMA_25'] = SMA(close_price, timeperiod=25)

        # Apply Bollinger Bands (BBANDS) using close prices
        df['BB_upper'], df['BB_middle'], df['BB_lower'] = BBANDS(close_price, timeperiod=20, nbdevup=2, nbdevdn=2, matype=0)

        # Apply Stochastic Oscillator (STOCH) using high, low, and close prices
        # Assuming 'high' and 'low' are same as 'ltp' for this example
        high = low = ltp
        df['slowk'], df['slowd'] = STOCH(high, low, close_price, fastk_period=5, slowk_period=3, slowk_matype=0, slowd_period=3, slowd_matype=0)

        return df

    def start_fetching(self):
        logging.info("Starting data fetching...")
        data = self.fetch_data()

        # Check for successful status and presence of 'data' key
        if data.get('status') == 'success' and 'data' in data:
            logging.debug(f"Data fetched successfully: {data}")

            # Extract data into a DataFrame
            rows = []
            for item in data['data']:
                call_data = item['call_options']['market_data']
                put_data = item['put_options']['market_data']

                rows.append({
                    'instrument': 'call',
                    'ltp': call_data['ltp'],
                    'volume': call_data['volume'],
                    'oi': call_data['oi'],
                    'close_price': call_data['close_price']
                })
                rows.append({
                    'instrument': 'put',
                    'ltp': put_data['ltp'],
                    'volume': put_data['volume'],
                    'oi': put_data['oi'],
                    'close_price': put_data['close_price']
                })

            df = pd.DataFrame(rows)

            # Apply indicators
            df = self.apply_indicators(df)

            # Print DataFrame to log
            logging.debug(f"Processed DataFrame with indicators:\n{df.head()}")

            # Here you can add logic to store df to the database
            # self.store_to_db(df)

        else:
            logging.error(f"API error or unexpected response format: {data}")

In [7]:
fetcher = OptionChainFetcher()
fetcher.start_fetching()

In [1]:
import upstox_client
from upstox_client.rest import ApiException
import websocket
import threading
import json
import time
from api import (apikey_his, apisecret_his)

# Load Access Token from file
with open('api/token/accessToken_oc.txt', 'r') as token_file:
    ACCESS_TOKEN = token_file.read().strip()

# Load instruments data from JSON file
with open('api/instrument/nse.json', 'r') as instrument_file:
    instruments_data = json.load(instrument_file)

# Filter criteria
criteria = {"segment": "NSE_EQ", "instrument_type": "EQ"}
filtered_instruments = [
    instr for instr in instruments_data
    if instr.get("segment") == criteria["segment"]
    and instr.get("instrument_type") == criteria["instrument_type"]
]

# Extract the instrument_key from the filtered instruments
if filtered_instruments:
    instrument_key = filtered_instruments[0].get('instrument_key')
    print(f"Instrument Key: {instrument_key}")
else:
    print("No instruments matched the criteria.")
    exit(1)

# API version and WebSocket settings
API_VERSION = '2.0'
PING_INTERVAL = 30
WEBSOCKET_URL = f"wss://websocket_url?access_token={ACCESS_TOKEN}&api_key={apikey_his}&version={API_VERSION}"

# WebSocket functions
def on_message(ws, message):
    data = json.loads(message)
    print("Received data:", data)

def on_error(ws, error):
    print(f"WebSocket error: {error}")

def on_close(ws, close_status_code, close_msg):
    print("WebSocket connection closed")

def on_open(ws):
    print("WebSocket connection opened")
    
    # Subscribe to live data for the instrument_key
    subscribe_payload = {
        "type": "subscribe",
        "instrument_key": instrument_key,
        "interval": "1minute"
    }
    ws.send(json.dumps(subscribe_payload))

    # Start ping-pong mechanism
    def send_ping():
        while True:
            if ws.sock and ws.sock.connected:
                ws.send(json.dumps({"type": "ping"}))
                print("Sent ping")
            time.sleep(PING_INTERVAL)

    ping_thread = threading.Thread(target=send_ping)
    ping_thread.daemon = True
    ping_thread.start()

# WebSocket runner
def run_websocket():
    ws = websocket.WebSocketApp(WEBSOCKET_URL,
                                on_message=on_message,
                                on_error=on_error,
                                on_close=on_close,
                                on_open=on_open)
    ws.run_forever(ping_interval=PING_INTERVAL)

if __name__ == "__main__":
    # Start WebSocket
    run_websocket()


FileNotFoundError: [Errno 2] No such file or directory: 'api/token/accessToken_oc.txt'

In [12]:
import requests
import pyotp
import os
import json

# Import required keys from api module
from api import (
    apikey_his, apisecret_his, redirect_url, totp_secret
)

def generate_totp_code(totp_secret):
    """
    Generates a TOTP code using the provided secret.
    """
    try:
        totp = pyotp.TOTP(totp_secret)
        generated_code = totp.now()
        print(f"[DEBUG] Generated TOTP code: {generated_code}")
        return generated_code
    except Exception as e:
        print(f"[ERROR] Failed to generate TOTP: {e}")
        raise

def get_access_token():
    """
    Fetches and saves the access token using Upstox API.
    """
    try:
        url = 'https://api.upstox.com/v2/login/authorization/token'
        headers = {
            'accept': 'application/json',
            'Content-Type': 'application/x-www-form-urlencoded'
        }

        # Generate TOTP code for authentication
        totp_code = generate_totp_code(totp_secret)
        print("[DEBUG] TOTP Code generated successfully")

        # Use the TOTP code as the auth_code
        code = totp_code
        client_id = apikey_his
        client_secret = apisecret_his
        redirect_uri = redirect_url
        grant_type = 'authorization_code'

        # Construct the payload
        data = {
            'code': code,
            'client_id': client_id,
            'client_secret': client_secret,
            'redirect_uri': redirect_uri,
            'grant_type': grant_type
        }

        print(f"[DEBUG] Request Data: {data}")

        # Send the POST request to fetch the access token
        response = requests.post(url, headers=headers, data=data)

        if response.status_code == 200:
            # Extract the access_token from the response
            api_response = response.json()
            access_token = api_response.get('access_token')

            # Create the directory if it doesn't exist
            token_dir = "api/token"
            if not os.path.exists(token_dir):
                os.makedirs(token_dir)

            # Save the access_token to a file
            with open(os.path.join(token_dir, "accessToken_his.txt"), "w") as token_file:
                token_file.write(access_token)
            print("[DEBUG] Access token saved successfully in 'accessToken_his.txt'")

            # Fetch user profile using the access token
            fetch_user_profile(access_token)
        else:
            print(f"[ERROR] Failed to obtain access token: {response.text}")

    except Exception as e:
        print(f"[ERROR] Unexpected error occurred: {e}")

def fetch_user_profile(access_token):
    """
    Fetches and saves the user's profile using the access token.
    """
    try:
        url = 'https://api.upstox.com/v2/user/profile'
        headers = {
            'Authorization': f'Bearer {access_token}',
            'accept': 'application/json'
        }

        print("[DEBUG] Sending request to fetch user profile...")

        # Make the request to fetch user profile
        response = requests.get(url, headers=headers)
        
        if response.status_code == 200:
            profile_data = response.json()
            print(f"[DEBUG] User profile fetched successfully: {json.dumps(profile_data, indent=2)}")
            
            # Save the user profile data to a file
            with open(os.path.join("api/token", "profileData.json"), "w") as profile_file:
                json.dump(profile_data, profile_file, indent=2)
            print("[DEBUG] User profile saved successfully in 'profileData.json'")
        else:
            print(f"[ERROR] Failed to fetch user profile: {response.text}")

    except Exception as e:
        print(f"[ERROR] Unexpected error occurred while fetching profile: {e}")

# Run the function to fetch and save the access token and user profile
get_access_token()


[DEBUG] Generated TOTP code: 116968
[DEBUG] TOTP Code generated successfully
[DEBUG] Request Data: {'code': '116968', 'client_id': '096449e9-13fb-4089-982c-82748d231d1a', 'client_secret': 'wkq01jsw2c', 'redirect_uri': 'https://127.0.0.1:50000/', 'grant_type': 'authorization_code'}
[ERROR] Failed to obtain access token: {"status":"error","errors":[{"errorCode":"UDAPI100500","message":"Something went wrong... please contact us","propertyPath":null,"invalidValue":null,"error_code":"UDAPI100500","property_path":null,"invalid_value":null}]}
